# TabPFN 2.5 — Train & Export for Inference Agent

TabPFN uses in-context learning: training data is bundled and the model
"fits" in ~0.2s at inference time. This notebook:

1. Runs 5-fold stratified CV per disease for robust evaluation
2. Exports the best fold's training data as a pickle bundle for the agent

**Diseases:** Hypertension, Stroke, CHD, Alzheimer's
**Dataset:** Synthea Coherent (151 patients, combined modality)

## 0. Setup

In [ ]:
import sys, time, pickle, json, logging, warnings
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tabpfn import TabPFNClassifier

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# ═══ CONFIGURATION ═══
LOCAL_DATA_DIR = "../data/raw/preprocessing"
LOCAL_EXPORT_DIR = Path("../models/exported")
LOCAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

S3_BUCKET = "guidance-multimodal-hcls-healthai-machinelearning"
S3_MODEL_PREFIX = "models"
UPLOAD_TO_S3 = False
N_FOLDS = 5

DISEASE_CODES = {
    'alzheimers_prediction': '26929004',
    'coronary_heart_disease_prediction': '53741008',
    'stroke_prediction': '230690007',
    'hypertension_prediction': '59621000',
}
DISEASES = list(DISEASE_CODES.keys())

OPTIMAL_SEEDS = {
    'alzheimers_prediction': 36,
    'coronary_heart_disease_prediction': 2,
    'stroke_prediction': 30,
    'hypertension_prediction': 11,
}

BENCHMARK_F1 = {
    "hypertension_prediction": 0.833,
    "stroke_prediction": 1.000,
    "coronary_heart_disease_prediction": 0.000,
    "alzheimers_prediction": 0.857,
}

print(f"TabPFN {N_FOLDS}-Fold CV")
print(f"Diseases: {DISEASES}")

## 1. Load Data

In [ ]:
sys.path.insert(0, '..')
from src.data.preprocessing_v2 import load_and_preprocess_all, build_combined_dataset

modalities = load_and_preprocess_all(LOCAL_DATA_DIR)
combined = build_combined_dataset(modalities)
print(f"Dataset: {combined.shape[0]} patients, {combined.shape[1]} features")

for d in DISEASES:
    n_pos = (combined[d] == '1').sum()
    print(f"  {d}: {n_pos}/{len(combined)} ({n_pos/len(combined)*100:.1f}%)")

## 2. Encoding Helper

In [ ]:
def encode_for_tabpfn(train, test, target):
    """Encode string features to numeric for TabPFN. Matches benchmark."""
    X_train = train.drop(columns=[target]).copy()
    y_train = train[target].astype(int).values
    X_test = test.drop(columns=[target]).copy()
    y_test = test[target].astype(int).values

    column_names = list(X_train.columns)
    cat_indices = []
    encoders = {}

    for i, col in enumerate(X_train.columns):
        try:
            X_train[col] = pd.to_numeric(X_train[col])
            X_test[col] = pd.to_numeric(X_test[col])
        except (ValueError, TypeError):
            le = LabelEncoder()
            all_vals = pd.concat([X_train[col], X_test[col]]).astype(str)
            le.fit(all_vals)
            X_train[col] = le.transform(X_train[col].astype(str))
            X_test[col] = le.transform(X_test[col].astype(str))
            cat_indices.append(i)
            encoders[col] = le

    X_train = X_train.fillna(0).astype(float)
    X_test = X_test.fillna(0).astype(float)
    return X_train.values, y_train, X_test.values, y_test, cat_indices, encoders, column_names

def prepare_cv_dataset(dataset, target):
    """Drop patientID + other targets for CV."""
    other_targets = [d for d in DISEASES if d != target]
    drop_cols = ['patientID'] + other_targets
    drop_cols = [c for c in drop_cols if c in dataset.columns]
    return dataset.drop(columns=drop_cols, errors='ignore')

## 3. 5-Fold Stratified Cross-Validation
Runs CV per disease, exports the best fold's training data as the inference bundle.

In [ ]:
cv_results = {}
results = {}
bundles = {}

for disease in DISEASES:
    seed = OPTIMAL_SEEDS[disease]
    cv_data = prepare_cv_dataset(combined, disease)
    y_all = cv_data[disease].astype(int).values

    print(f"\n{'='*60}")
    print(f"TabPFN {N_FOLDS}-Fold CV: {disease} (seed={seed})")
    print(f"{'='*60}")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold_results = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(cv_data, y_all)):
        train_fold = cv_data.iloc[train_idx]
        test_fold = cv_data.iloc[test_idx]

        X_train, y_train, X_test, y_test, cat_indices, encoders, col_names = \
            encode_for_tabpfn(train_fold, test_fold, disease)

        start = time.time()
        clf = TabPFNClassifier(
            random_state=42,
            categorical_features_indices=cat_indices if cat_indices else None,
            ignore_pretraining_limits=True,
        )
        clf.fit(X_train, y_train)
        fit_time = time.time() - start

        y_pred = clf.predict(X_test)
        fold_f1 = round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4)
        fold_acc = round(accuracy_score(y_test, y_pred), 4)
        n_pos = int(y_test.sum())

        fold_results.append({
            'fold': fold_idx + 1, 'f1': fold_f1, 'accuracy': fold_acc,
            'precision': round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'recall': round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'fit_time': round(fit_time, 2), 'n_pos_test': n_pos,
            'X_train': X_train, 'y_train': y_train,
            'cat_indices': cat_indices, 'encoders': encoders, 'col_names': col_names,
        })
        print(f"  Fold {fold_idx+1}: F1={fold_f1:.3f}  Acc={fold_acc:.3f}  pos={n_pos}/{len(test_idx)}")

    f1s = [r['f1'] for r in fold_results]
    mean_f1 = round(np.mean(f1s), 4)
    std_f1 = round(np.std(f1s), 4)
    best = max(fold_results, key=lambda r: r['f1'])

    cv_results[disease] = {'mean_f1': mean_f1, 'std_f1': std_f1, 'best_fold': best['fold'], 'best_f1': best['f1']}
    results[disease] = {'f1': best['f1'], 'cv_mean_f1': mean_f1, 'cv_std_f1': std_f1}

    print(f"\n  CV Mean: {mean_f1} ± {std_f1} | Best fold {best['fold']}: F1={best['f1']:.3f}")

    # Export best fold bundle
    bundle = {
        'X_train': best['X_train'], 'y_train': best['y_train'],
        'cat_indices': best['cat_indices'], 'encoders': best['encoders'],
        'column_names': best['col_names'], 'disease': disease,
        'seed': seed, 'n_samples': len(best['X_train']),
        'n_features': best['X_train'].shape[1],
        'cv_mean_f1': mean_f1, 'cv_std_f1': std_f1,
        'best_fold': best['fold'], 'n_folds': N_FOLDS,
        'created_at': datetime.utcnow().isoformat(),
    }
    bundles[disease] = bundle

    bundle_dir = LOCAL_EXPORT_DIR / "tabpfn" / disease
    bundle_dir.mkdir(parents=True, exist_ok=True)
    with open(bundle_dir / "training_data.pkl", 'wb') as f:
        pickle.dump(bundle, f)
    print(f"  Exported to: {bundle_dir / 'training_data.pkl'}")

## 4. Summary

In [ ]:
print(f"\n{'='*60}")
print(f"TabPFN {N_FOLDS}-Fold CV Summary")
print(f"{'='*60}")
print(f"{'Disease':<40} {'CV Mean':>8} {'± Std':>7} {'Best':>6} {'Benchmark':>10}")
print("-" * 71)
for d in DISEASES:
    cv = cv_results[d]
    bm = BENCHMARK_F1[d]
    print(f"  {d:<38} {cv['mean_f1']:>6.3f}  ±{cv['std_f1']:.3f}  {cv['best_f1']:>5.3f}  {bm:>8.3f}")

## 5. Upload to S3

In [ ]:
if UPLOAD_TO_S3:
    import boto3
    s3 = boto3.client('s3')
    for disease in DISEASES:
        local_file = LOCAL_EXPORT_DIR / "tabpfn" / disease / "training_data.pkl"
        s3_key = f"{S3_MODEL_PREFIX}/tabpfn/{disease}/training_data.pkl"
        s3.upload_file(str(local_file), S3_BUCKET, s3_key)
        print(f"  {disease} → s3://{S3_BUCKET}/{s3_key}")
else:
    print(f"S3 upload skipped. Bundles at: {LOCAL_EXPORT_DIR.resolve()}")

## How the Agent Uses These Bundles

At inference time:
```python
bundle = pickle.load(open("training_data.pkl", "rb"))
clf = TabPFNClassifier(categorical_features_indices=bundle['cat_indices'])
clf.fit(bundle['X_train'], bundle['y_train'])  # ~0.2s
proba = clf.predict_proba(patient_feature_vector)
```